<a href="https://colab.research.google.com/github/Ladecodes1/formative-3-group/blob/main/part4_gradient_descent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Part 4: Gradient Descent in Code
Goal: turn the Part 3 manual calculations into Python and check that the code gives the same numbers.
Model: y = m₁x₁ + m₂x₂ + b, written in matrix form as ŷ = Xm + b.
Settings agreed by the group: learning rate α = 0.005, 3 updates, one shared bias b, rounded to 4 decimal places.

## Step 1: Imports and data
Everything is stored as NumPy arrays so the maths uses matrix multiplication, as the brief requires.
Each row of X is one data point and each column is one feature (x₁, x₂).

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import approx_fprime

X = np.array([[1, 3],
              [4, 10]], dtype=float)
y = np.array([5, 6], dtype=float)

m_initial = np.array([-1, 2], dtype=float)
b_initial = np.array([1, 1], dtype=float)  # one shared bias, written once per data point

LEARNING_RATE = 0.005
N_UPDATES = 3
n = len(y)

print("X =\n", X)
print("y =", y)
print("Initial m =", m_initial)
print("Initial b =", b_initial)
print("Learning rate:", LEARNING_RATE, "| Updates:", N_UPDATES, "| Data points n:", n)

X =
 [[ 1.  3.]
 [ 4. 10.]]
y = [5. 6.]
Initial m = [-1.  2.]
Initial b = [1. 1.]
Learning rate: 0.005 | Updates: 3 | Data points n: 2


## Step 2: Prediction and cost
`predict` computes ŷ = Xm + b with one matrix multiplication.
`mse` computes the cost J = (1/n) · eᵀe, where e = ŷ − y is the error for each data point.

In [2]:
def predict(X, m, b):
    """Predicted values: y_hat = Xm + b."""
    return X @ m + b


def mse(X, y, m, b):
    """Mean squared error: J = (1/n) * e^T e, where e = y_hat - y."""
    errors = predict(X, m, b) - y
    return (errors @ errors) / len(y)


y_hat = predict(X, m_initial, b_initial)
print("Predictions y_hat:", y_hat)
print("Errors y_hat - y: ", y_hat - y)
print("Cost J:           ", mse(X, y, m_initial, b_initial))

Predictions y_hat: [ 6. 17.]
Errors y_hat - y:  [ 1. 11.]
Cost J:            61.0


These match Update 1 in the Part 3 manual calculations: ŷ = [6, 17], e = [1, 11], J = 61.

## Step 3: Derivatives with SciPy
`derivative` accepts any equation (a Python function of the parameters) and uses SciPy's `approx_fprime` to compute its derivative numerically.
The equation we pass in is the cost J(m₁, m₂, b). The bias is one shared value, so it is a single parameter here.

In [3]:
def derivative(equation, params):
    """Numerical derivative of an equation with respect to each of its parameters (SciPy)."""
    return approx_fprime(params, equation)


def cost_equation(params):
    """The cost J as an equation of the parameters [m1, m2, b]."""
    m, b = params[:2], np.full(n, params[2])
    return mse(X, y, m, b)


params_initial = np.array([*m_initial, b_initial[0]])
scipy_gradient = derivative(cost_equation, params_initial)

print("SciPy dJ/dm1, dJ/dm2, dJ/db:", np.round(scipy_gradient, 4))

SciPy dJ/dm1, dJ/dm2, dJ/db: [ 45. 113.  12.]


We check SciPy against the formulas derived by hand in Part 3:
∂J/∂m = (2/n) · Xᵀ(ŷ − y) and ∂J/∂b = (2/n) · Σ(ŷ − y).

In [4]:
def gradients(X, y, m, b):
    """Gradients from the Part 3 derivation: dJ/dm = (2/n) X^T e and dJ/db = (2/n) sum(e)."""
    errors = predict(X, m, b) - y
    grad_m = (2 / len(y)) * X.T @ errors
    grad_b = (2 / len(y)) * errors.sum()
    return grad_m, grad_b


grad_m, grad_b = gradients(X, y, m_initial, b_initial)
print("Formula dJ/dm:", grad_m, "| dJ/db:", grad_b)
print("SciPy agrees with the formula:", np.allclose(scipy_gradient, [*grad_m, grad_b], atol=1e-4))

Formula dJ/dm: [ 45. 113.] | dJ/db: 12.0
SciPy agrees with the formula: True


Both give ∂J/∂m = [45, 113] and ∂J/∂b = 12, matching Update 1 in Part 3.